## **Step 1: Imports**

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

## **Step 2: Load Dataset**

In [ ]:
df = pd.read_csv(
    "soc_analyst_dataset_200k.csv"
)

print(df.shape)

(200000, 14)


## **Step 3: Target Variable**

In [ ]:
target = "response_action"

## **Step 4: Encode Target**

In [ ]:
encoder = LabelEncoder()

df[target] = encoder.fit_transform(
    df[target]
)

## **Step 5: Feature Selection**

Drop only target

In [ ]:
drop_cols = [
    "alert_id",
    "response_action"
]

X = df.drop(
    columns=drop_cols
)

y = df[target]

Step 6: Split Features

In [ ]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_cols = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

Step 7: Preprocessing

Numeric

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

Categorical

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

## **Step 8: Column Transformer**

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_cols
        ),
        (
            "cat",
            categorical_transformer,
            categorical_cols
        )
    ]
)

## **Step 9: Train/Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

## **Step 10: Model**

In [ ]:
response_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

## **Step 11: Build Pipeline**

In [ ]:
pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            response_model
        )
    ]
)

## **Step 12: Train**

In [ ]:
pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['severity_raw',
                                                   'failed_login_count',
                                                   'geo_anomaly',
                                                   'known_ioc_match',
                                                   'malware_detected',
                                                   'user_risk_score',
                                                   'overall_risk_score']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['alert_source', 'alert_type',
                                                   'asset_criticality',
                                                   'threat_label',
                                                   'priority_label'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=20, n_estimators=400,
                                        n_jobs=-1, random_state=42))])

## **Step 13: Evaluate**

In [ ]:
y_pred = pipeline.predict(
    X_test
)

print(
    accuracy_score(
        y_test,
        y_pred
    )
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2370
           1       1.00      1.00      1.00     27113
           2       1.00      1.00      1.00       463
           3       1.00      1.00      1.00     10054

    accuracy                           1.00     40000
   macro avg       1.00      1.00      1.00     40000
weighted avg       1.00      1.00      1.00     40000

[[ 2370     0     0     0]
 [    0 27113     0     0]
 [    0     0   463     0]
 [    0     0     0 10054]]


## **Step 14: Save Model**

In [ ]:
joblib.dump(
    pipeline,
    "response_model.pkl"
)

['response_model.pkl']

## **Step 15: Save Encoder**

In [ ]:
joblib.dump(
    encoder,
    "response_encoder.pkl"
)

['response_encoder.pkl']

## **Step 16: Verify**

In [ ]:
import os

print(
    os.path.exists(
        "response_model.pkl"
    )
)

print(
    os.path.exists(
        "response_encoder.pkl"
    )
)

True
True


## **Step 17: Download**

In [ ]:
from google.colab import files

files.download(
    "response_model.pkl"
)

files.download(
    "response_encoder.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **Testing**

In [ ]:
sample = X_test.iloc[[0]]

prediction = pipeline.predict(
    sample
)

action = encoder.inverse_transform(
    prediction
)

print(action)

['Investigate Endpoint']


In [ ]:
print(X.columns.tolist())

['alert_source', 'alert_type', 'severity_raw', 'failed_login_count', 'geo_anomaly', 'known_ioc_match', 'malware_detected', 'asset_criticality', 'user_risk_score', 'overall_risk_score', 'threat_label', 'priority_label']
